# Structured Output

### Structured output allows agents to return data in a specific, predictable format. Instead of parsing natural language responses, you get structured data in the form of JSON objects, Pydantic models, or dataclasses that your application can use directly.

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

os.environ["OLLAMA_MODEL"] = os.getenv("OLLAMA_MODEL", "Gemma4")

model = init_chat_model(os.environ["OLLAMA_MODEL"], model_kwargs={"temperature": 0.7, "max_tokens": 1000})

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    director: str = Field(description="The director of the movie")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="The movies rating out of 10")

In [7]:
model_with_structure = model.with_structured_output(Movie)

model_with_structure

_ChatModelBinding(bound=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, model='Gemma4'), kwargs={'format': {'properties': {'title': {'description': 'The title of the movie', 'title': 'Title', 'type': 'string'}, 'year': {'description': 'The release year of the movie', 'title': 'Year', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'title': 'Director', 'type': 'string'}, 'genre': {'description': 'The genre of the movie', 'title': 'Genre', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'title': 'Rating', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'genre', 'rating'], 'title': 'Movie', 'type': 'object'}, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema'}, 'schema': <class '__main__.Movie'>}}, config={}, config_factories=[])
| PydanticOutputParser(pydantic_object=<class '__main__.Movie'>)

In [8]:
response = model_with_structure.invoke("Provide details about the movie Inception")

In [9]:
print(response)

title='Inception' year=2010 director='Christopher Nolan' genre='Science Fiction / Action Thriller / Neo-Noir' rating=4.5


### Messaged output alongside parsed data

In [10]:
model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")

response

{'raw': AIMessage(content='{ "title": "Inception", "year": 2010, "director": "Christopher Nolan", "genre": "Sci-Fi Thriller, Action, Crime Drama", "rating": 4.5 }', additional_kwargs={}, response_metadata={'model': 'Gemma4', 'created_at': '2026-08-08T10:44:01.130018Z', 'done': True, 'done_reason': 'stop', 'total_duration': 24405470709, 'load_duration': 242639959, 'prompt_eval_count': 670, 'prompt_eval_duration': 168662000, 'eval_count': 49, 'eval_duration': 1745105000, 'logprobs': None, 'model_name': 'Gemma4', 'model_provider': 'ollama'}, id='lc_run--019fe0f8-9a93-70c1-a8a9-5aa76ca0d441-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 670, 'output_tokens': 49, 'total_tokens': 719}),
 'parsed': Movie(title='Inception', year=2010, director='Christopher Nolan', genre='Sci-Fi Thriller, Action, Crime Drama', rating=4.5),
 'parsing_error': None}

# Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, desctiption="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")

response


/var/folders/pb/7ffdf5v91cnghvkx6p8m5pkw0000gn/T/ipykernel_24318/1159941599.py:12: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desctiption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  budget: float | None = Field(None, desctiption="Budget in millions USD")


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role="Dominick 'Dom' Cobb"), Actor(name='Joseph Gordon-Levison', role='Arthur Shelby Jr.'), Actor(name='Elliot Page', role='Ariadne Standish'), Actor(name='Tom Hardy', role='Eames Barnesker'), Actor(name='Ken Watanabe', role='Saito')], genres=['Sci-Fi', 'Action Thriller', 'Crime'], budget=160000000.0)

# TypedDict

##### TypedDict provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation.

In [15]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_structure = model.with_structured_output(MovieDict)

response = model_with_structure.invoke("Provide details about the movie The Avengers")

response

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 7.85}

# DataClasses

##### A data class is a class typically containing mainly data, through there aren't really any restrictions. You create it using the @dataclass

In [19]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model=os.environ["OLLAMA_MODEL"],
    response_format=ContactInfo
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact information from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [20]:
# Data class
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=os.environ["OLLAMA_MODEL"],
    response_format=ContactInfo
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact information from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
